# Figures 1–2: one-dimensional cosine problem

Run all cells to generate `results/figure_01.png`, `results/figure_02.png`, and their CSV data.
The problem uses 1,000 unknowns, 50 DCT measurements, and 3% noise with seed 0.

GKS, S-GKS, and PS-GKS use 300 iterations; restarted and recycled S-GKS use 800.
MM uses $p=1$ and $\epsilon=10^{-3}$; IAS uses $r=-1$ and $\beta=1$.


In [ ]:
print("Device: CPU")

from pathlib import Path
import os, sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
# Use eight BLAS threads for the dense inverse and projected solves.
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
from threadpoolctl import threadpool_limits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiment_1d import make_problem, set_style, logged_solve
thread_limits = threadpool_limits(limits=8)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
set_style()
A, L, Linv, Lpinv_factory, b, ground_truth, noise_var = make_problem()
import operators
import scipy.sparse as sps
Linv = operators.MatrixLinearOperator(np.linalg.inv(L.A.toarray()))
def Lpinv_factory(wr):
    return Linv @ operators.MatrixLinearOperator(sps.diags(1.0 / wr))
n = len(ground_truth)
print(f"{n} unknowns, {len(b)} DCT measurements; relative noise 0.03; seed 0")


## Set up the solvers


In [ ]:
from msolver import VanillaGKS, MMGKS, IASGKS
# Apply after all numerical libraries are loaded, including SciPy's BLAS.
thread_limits = threadpool_limits(limits=8)
v_solver = VanillaGKS(A, L, b, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
mm_solver = MMGKS(A, L, b, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
ias_solver = IASGKS(A, L, b, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
n_its = 300
n_its_recres = 800
common = dict(init_dim=5, regparam="dp", x_true=ground_truth,
              lambdah_min=1e-7, lambdah_max=1e7, regparam_kwargs={"eta":1.01})
results = {}


## Compute GKS and the two families of weighted reconstructions


In [ ]:
results[("V", "GKS")] = logged_solve(OUT / "intro_solver.log", "V GKS", v_solver.solve,
    n_iter=n_its, x0=None, lambdah0=1.0, weight_params={}, ksparsity=None, xker=None, **common)
v = results[("V", "GKS")]
warm = dict(x0=v["x"], lambdah0=v["lambdah"])
for weights, solver, params in [
    ("MM", mm_solver, {"q":1.0, "epsilon":1e-3}),
    ("IAS", ias_solver, {"hyperpr_r":-1.0, "hyperpr_beta":1.0})]:
    for method, iterations, options in [
        ("S-GKS", n_its, {}),
        ("resS-GKS", n_its_recres, {"restart_dim":40, "ksparsity":4}),
        ("recS-GKS", n_its_recres, {"min_dim":20, "max_dim":40})]:
        results[(weights, method)] = logged_solve(OUT / "intro_solver.log", weights+" "+method,
            solver.solve, n_iter=iterations, weight_params=params, **warm, **common, **options)
    results[(weights, "PS-GKS")] = logged_solve(OUT / "intro_solver.log", weights+" PS-GKS",
        solver.solve_with_priorconditioning, n_iter=n_its, weight_params=params, **warm, **common)


## Figure 1: final reconstructions


In [ ]:
method_colors = {"GKS":"C1", "S-GKS":"C4", "resS-GKS":"C4", "recS-GKS":"C4",
                 "PS-GKS":"C2", "resPS-GKS":"C2", "recPS-GKS":"C2", "PS-GKB":"C5",
                 "FGK":"C3", "FLSQR-I":"C3", "FLSQR-R":"C3", "FLSQR-W":"C3", "IRW-FLSQR":"C3"}
method_lines = {"GKS":"solid", "S-GKS":"solid", "resS-GKS":"dotted", "recS-GKS":"dashed",
                "PS-GKS":"solid", "resPS-GKS":"dotted", "recPS-GKS":"dashed", "PS-GKB":"dashed",
                "FGK":"dotted", "FLSQR-I":"dotted", "FLSQR-R":"dashed", "FLSQR-W":"dashdot",
                "IRW-FLSQR":(0, (3, 1, 1, 1, 1, 1))}

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib import patches
plt.rcParams.update({"font.size":14})
fig, axs = plt.subplots(2, 1, figsize=(14,8))
index = np.arange(n)
rows = []
for row, weights in enumerate(["MM", "IAS"]):
    ax = axs[row]
    ax.plot(index, ground_truth, color="C9", label="Ground truth")
    zoom = inset_axes(ax, width="30%", height="30%", bbox_to_anchor=(0.13,-0.78,0.9,2.0),
                      bbox_transform=ax.transAxes, loc="center")
    zoom.set(xlim=(370,430), ylim=(-1,5), xticks=[], yticks=[])
    ax.add_patch(patches.Rectangle((370,-1),60,6,linewidth=0.5,edgecolor="gray",facecolor="none",linestyle="--"))
    mark_inset(ax, zoom, loc1=2, loc2=3, fc="none", ec="0.5")
    zoom.plot(index, ground_truth, color="C9")
    for method in ["GKS", "S-GKS", "PS-GKS"]:
        x = results[("V" if method == "GKS" else weights, method)]["x"]
        for target in (ax,zoom):
            target.plot(index, x, color=method_colors[method], lw=1)
            size = 12 if target is zoom and (method == "PS-GKS" or (weights == "IAS" and method == "GKS")) else 3
            target.scatter(index, x, color=method_colors[method], s=size, label=method)
        rows.extend({"weights":weights,"method":method,"index":int(i),"reconstruction":float(xi),
                     "ground_truth":float(ground_truth[i])} for i,xi in enumerate(x))
    ax.set(title=f"Reconstructions ({weights} weights)", xlabel="Index")
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "figure_01.png", dpi=250, bbox_inches="tight")
pd.DataFrame(rows).to_csv(OUT / "figure_01.csv", index=False)
plt.show()


## Figure 2: convergence and conditioning histories


In [ ]:
plt.rcParams.update({"font.size": 14})
fig, axs = plt.subplots(2,3,figsize=(14,8))
metrics = [("rre_history","RRE history"),("gini_indices","Gini index"),
           ("cond_ests",r"$\kappa$ of projected problem")]
rows = []
for row, weights in enumerate(["MM", "IAS"]):
    for col, (metric,title) in enumerate(metrics):
        ax = axs[row,col]
        for method in ["GKS","S-GKS","resS-GKS","recS-GKS","PS-GKS"]:
            values = np.asarray(results[("V" if method=="GKS" else weights,method)][metric])
            start = 5
            ax.plot(np.arange(start,len(values)),values[start:],color=method_colors[method],
                    ls=method_lines[method],lw=3,label=method)
            rows.extend({"weights":weights,"method":method,"metric":metric,"iteration_index":i,
                         "value":float(values[i])} for i in range(start,len(values)))
        ax.set(title=f"{title} ({weights})",xlabel="Iteration index")
        if metric=="cond_ests": ax.set_yscale("log")
        ax.legend(loc="lower right" if metric=="cond_ests" or (row==0 and col==0) else "best")
fig.tight_layout()
fig.savefig(OUT / "figure_02.png",dpi=250)
pd.DataFrame(rows).to_csv(OUT / "figure_02.csv",index=False)
plt.show()
